# Data Collection Notebook

This notebook collects public data from all sources for the Broadway MDS-SV analysis.

**Sources:**
- Instagram (public profiles)
- TikTok (hashtag data)
- Reddit (r/Broadway posts and comments)
- Google Trends
- BroadwayWorld grosses
- Ticket prices (SeatGeek/TodayTix)
- BroadwayBox discounts

**Output:** Raw data saved to `data/raw/` directories

In [ ]:
import sys
sys.path.append('..')

import yaml
import pandas as pd
from pathlib import Path
from datetime import datetime, timedelta
from tqdm import tqdm

# Import scrapers
from src.scrapers.instagram import InstagramScraper
from src.scrapers.tiktok import TikTokScraper
from src.scrapers.reddit import RedditScraper
from src.scrapers.trends import GoogleTrendsScraper
from src.scrapers.grosses import BroadwayWorldGrossesScraper
from src.scrapers.prices import TicketPriceScraper
from src.scrapers.discounts import BroadwayBoxScraper

In [ ]:
# Load configuration
with open('../config/shows.yaml', 'r') as f:
    config = yaml.safe_load(f)

shows = config['shows']
collection_settings = config['collection']

print(f"Loaded configuration for {len(shows)} shows")
print(f"CMM depth cohort: {[s['name'] for s in shows.values() if s['cohort'] == 'cmm_depth']}")
print(f"Broad appeal cohort: {[s['name'] for s in shows.values() if s['cohort'] == 'broad_appeal']}")

## 1. Instagram Data Collection

In [ ]:
ig_scraper = InstagramScraper(
    cache_dir=Path('../data/raw/instagram'),
    rate_limit=collection_settings['rate_limits']['instagram']
)

instagram_data = {}

for show_id, show_info in tqdm(shows.items(), desc="Scraping Instagram"):
    handle = show_info.get('instagram_handle')
    if not handle:
        print(f"Skipping {show_info['name']}: no Instagram handle")
        continue
    
    # Define date range (2 years back)
    end_date = datetime.now()
    start_date = end_date - timedelta(days=730)
    
    print(f"\nScraping Instagram for {show_info['name']} (@{handle})...")
    posts_df = ig_scraper.scrape_profile_posts(handle, start_date, end_date)
    stats = ig_scraper.get_profile_stats(handle)
    
    instagram_data[show_info['name']] = {
        'posts': posts_df,
        'stats': stats
    }
    
    print(f"  Collected {len(posts_df)} posts, {stats['followers']:,} followers")

print(f"\nInstagram collection complete. Collected data for {len(instagram_data)} shows.")

## 2. TikTok Data Collection

In [ ]:
tt_scraper = TikTokScraper(
    cache_dir=Path('../data/raw/tiktok'),
    rate_limit=collection_settings['rate_limits']['tiktok']
)

tiktok_data = {}

for show_id, show_info in tqdm(shows.items(), desc="Scraping TikTok"):
    hashtags = show_info.get('tiktok_hashtags', [])
    if not hashtags:
        print(f"Skipping {show_info['name']}: no TikTok hashtags")
        continue
    
    end_date = datetime.now()
    start_date = end_date - timedelta(days=365)
    
    print(f"\nScraping TikTok for {show_info['name']} ({hashtags})...")
    videos_df = tt_scraper.scrape_hashtag_videos(hashtags, start_date, end_date)
    
    tiktok_data[show_info['name']] = {'videos': videos_df}
    
    print(f"  Collected {len(videos_df)} videos")

print(f"\nTikTok collection complete. Collected data for {len(tiktok_data)} shows.")

## 3. Reddit Data Collection

In [ ]:
reddit_scraper = RedditScraper(
    cache_dir=Path('../data/raw/reddit'),
    rate_limit=collection_settings['rate_limits']['reddit']
)

reddit_data = {}

for show_id, show_info in tqdm(shows.items(), desc="Scraping Reddit"):
    queries = show_info.get('reddit_queries', [])
    if not queries:
        print(f"Skipping {show_info['name']}: no Reddit queries")
        continue
    
    end_date = datetime.now()
    start_date = end_date - timedelta(days=730)
    
    print(f"\nScraping Reddit for {show_info['name']} ({queries})...")
    activity = reddit_scraper.get_subreddit_activity(
        queries, subreddit='Broadway', start_date=start_date, end_date=end_date
    )
    
    reddit_data[show_info['name']] = activity
    
    print(f"  Collected {len(activity['posts'])} posts, {len(activity['comments'])} comments")

print(f"\nReddit collection complete. Collected data for {len(reddit_data)} shows.")

## 4. Google Trends Data Collection

In [ ]:
trends_scraper = GoogleTrendsScraper(
    cache_dir=Path('../data/raw/trends'),
    rate_limit=collection_settings['rate_limits']['trends']
)

trends_data = {}

for show_id, show_info in tqdm(shows.items(), desc="Fetching Google Trends"):
    show_name = show_info['name']
    queries = [show_name] + show_info.get('aliases', [])
    
    # Limit to first query if multiple
    query = queries[0]
    
    end_date = datetime.now()
    start_date = end_date - timedelta(days=365)
    
    print(f"\nFetching Google Trends for {show_name} (query: '{query}')...")
    trends_df = trends_scraper.fetch_interest_over_time(
        [query], start_date=start_date, end_date=end_date, geo='US', frequency='weekly'
    )
    
    trends_data[show_name] = trends_df
    
    if not trends_df.empty:
        print(f"  Collected {len(trends_df)} weeks of data")
    else:
        print(f"  No data available")

print(f"\nGoogle Trends collection complete. Collected data for {len(trends_data)} shows.")

## 5. BroadwayWorld Grosses Collection

In [ ]:
grosses_scraper = BroadwayWorldGrossesScraper(
    cache_dir=Path('../data/raw/grosses'),
    rate_limit=collection_settings['rate_limits']['grosses']
)

grosses_data = {}

for show_id, show_info in tqdm(shows.items(), desc="Scraping grosses"):
    show_name = show_info['name']
    
    print(f"\nScraping grosses for {show_name}...")
    grosses_df = grosses_scraper.scrape_show_grosses(show_name)
    
    grosses_data[show_name] = grosses_df
    
    print(f"  Collected {len(grosses_df)} weeks of grosses data")

print(f"\nGrosses collection complete. Collected data for {len(grosses_data)} shows.")

## 6. Ticket Prices Collection

In [ ]:
prices_scraper = TicketPriceScraper(
    cache_dir=Path('../data/raw/prices'),
    rate_limit=collection_settings['rate_limits']['prices']
)

prices_data = {}

for show_id, show_info in tqdm(shows.items(), desc="Scraping prices"):
    show_name = show_info['name']
    
    print(f"\nScraping ticket prices for {show_name}...")
    prices_df = prices_scraper.scrape_seatgeek_listings(show_name)
    
    prices_data[show_name] = prices_df
    
    if not prices_df.empty:
        print(f"  Collected price data: min=${prices_df['min_price'].iloc[0]:.2f}")
    else:
        print(f"  No price data available")

print(f"\nPrice collection complete. Collected data for {len(prices_data)} shows.")

## 7. BroadwayBox Discounts Collection

In [ ]:
discounts_scraper = BroadwayBoxScraper(
    cache_dir=Path('../data/raw/discounts'),
    rate_limit=collection_settings['rate_limits']['discounts']
)

discounts_data = {}

for show_id, show_info in tqdm(shows.items(), desc="Scraping discounts"):
    show_name = show_info['name']
    
    print(f"\nScraping discounts for {show_name}...")
    discounts_df = discounts_scraper.scrape_show_discounts(show_name)
    
    discounts_data[show_name] = discounts_df
    
    if not discounts_df.empty:
        print(f"  Found {discounts_df['discount_count'].iloc[0]} discount types")
    else:
        print(f"  No discount data available")

print(f"\nDiscount collection complete. Collected data for {len(discounts_data)} shows.")

## 8. Save Processed Data

In [ ]:
# Save all collected data to processed directory
processed_dir = Path('../data/processed')
processed_dir.mkdir(parents=True, exist_ok=True)

# Save Instagram data
for show_name, data in instagram_data.items():
    if not data['posts'].empty:
        data['posts'].to_csv(processed_dir / f'{show_name.replace(" ", "_")}_instagram.csv', index=False)

# Save TikTok data
for show_name, data in tiktok_data.items():
    if not data['videos'].empty:
        data['videos'].to_csv(processed_dir / f'{show_name.replace(" ", "_")}_tiktok.csv', index=False)

# Save Reddit data
for show_name, data in reddit_data.items():
    if not data['posts'].empty:
        data['posts'].to_csv(processed_dir / f'{show_name.replace(" ", "_")}_reddit_posts.csv', index=False)
    if not data['comments'].empty:
        data['comments'].to_csv(processed_dir / f'{show_name.replace(" ", "_")}_reddit_comments.csv', index=False)

# Save Trends data
for show_name, df in trends_data.items():
    if not df.empty:
        df.to_csv(processed_dir / f'{show_name.replace(" ", "_")}_trends.csv')

# Save grosses data
for show_name, df in grosses_data.items():
    if not df.empty:
        df.to_csv(processed_dir / f'{show_name.replace(" ", "_")}_grosses.csv', index=False)

# Save prices data
for show_name, df in prices_data.items():
    if not df.empty:
        df.to_csv(processed_dir / f'{show_name.replace(" ", "_")}_prices.csv', index=False)

# Save discounts data
for show_name, df in discounts_data.items():
    if not df.empty:
        df.to_csv(processed_dir / f'{show_name.replace(" ", "_")}_discounts.csv', index=False)

print(f"\nAll data saved to {processed_dir}/")
print("\nData collection complete!")

## Summary

This notebook collected public data from 7 sources. Next steps:
1. Run `02_build_scores.ipynb` to compute MDS and SV scores
2. Run `03_analysis.ipynb` to test the MDS-SV relationship